# Mushroom Edibility Classification – Practice Skeleton

**Short name (GitHub):** `MushEdib`  
**Lab source:** UCI *Mushroom* table (Audubon Society field-guide codes) + the attached Data Prep / EDA / RF / Evaluation brief.  
**Data:** `data/mushrooms.csv` (8,124 × 23 letter codes). Target `class`: **e** = edible, **p** = poisonous.  
**Companion files:** `MushEdib_Solution.ipynb`, `MushEdib_Reusable_Template.ipynb`, `MushEdib.py`, `MushEdib_Cheatsheet.docx`, `MushEdib_Project_Memo.docx`, `MushEdib_Strategy_Guide.docx`, `MushEdib_1Page_Summary_Report.docx`, `mushedib_flowchart.png`.

Work top to bottom. Cells marked `# YOUR CODE HERE` are for you. Peek at the solution notebook only after you have an answer.

**This is not a foraging app.** A perfect score on this 1987 codebook does not mean the next cap you pick is safe.

You will:

1. Clean `?` in `stalk-root` → `u`, check duplicates.
2. Plot class balance, odor × class, cap-color × class, and a 12-feature factorize heatmap.
3. Drop zero-variance `veil-type`, LabelEncode every column, 80/20 split (`random_state=42`).
4. Fit `RandomForestClassifier(random_state=42)` and evaluate.
5. Compare alternates, run extra practice, twist simulation knobs.



## Inline cheat-sheet (keep this cell visible)

See also **`MushEdib_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| Load | `pd.read_csv("data/mushrooms.csv")` |
| Missing in this table | literal `"?"`, almost only in `stalk-root` (2,480 rows) |
| Task replacement | `df["stalk-root"] = df["stalk-root"].replace("?", "u")` |
| Collision | official codebook already uses `u` = cup. Task still wants `"u"` for unknown |
| Duplicates | `df.duplicated().sum()` then `drop_duplicates()` |
| Zero-variance | `veil-type` is always `p` — drop before encoding |
| Factorize (EDA only) | `df[col], _ = pd.factorize(df[col])` — arbitrary integer codes |
| Top-12 heatmap | drop `class` *before* `.head(12)`; heatmap is 12×12, not 13×13 |
| Encode for trees | `LabelEncoder().fit_transform` per column |
| Encode for distance / LR | `pd.get_dummies` (one-hot) |
| Split | `train_test_split(X, y, test_size=0.2, random_state=42)` |
| RF | `RandomForestClassifier(random_state=42)` — defaults, 100 trees |
| Costly cell | actual **p**, predicted **e** (false edible) |
| Odor rule | majority class per odor code ≈ 98.5% on this table |
| Class map after LE | alphabetical → `e=0`, `p=1` |

**sklearn note.** Trees do not need scaling. LabelEncoder invents a fake order (`brown < buff < cinnamon`) that is fine for RF / DT and wrong for kNN / unpenalized linear models.



## Flowchart of the desired outcome

![MushEdib flow](mushedib_flowchart.png)

Clean first (unknown stalk-root, no dups). Look at odor before you fit anything. Drop `veil-type`. Freeze the 80/20 seed at 42. Score the model on the costly cell, not only accuracy. Then break it on purpose in the simulation section.



## 0. Packages


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    recall_score, f1_score,
)
from sklearn.feature_selection import mutual_info_classif

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# optional local helper
try:
    import MushEdib as me
except ImportError:
    me = None
print("ready")


## 1. Data preparation

Load `data/mushrooms.csv`. Display the first 5 rows. Call `.info()`. Count `"?"` in `stalk-root` and replace them with `"u"`. Count and drop exact duplicate rows. Print the cleaned shape and `class` counts.

Expected: 8,124 rows × 23 columns, 2,480 question marks, 0 duplicates, class split 4,208 edible / 3,916 poisonous.


In [ ]:
# YOUR CODE HERE
df = None
print("First 5 rows:")
# print(df.head())
print("DataFrame info:")
# df.info()
n_q = None
print("Missing ('?') values in 'stalk-root':", n_q)
# replace '?' with 'u'
n_dups = None
print("Number of duplicate rows:", n_dups)
print("Cleaned dataset shape:", None)
# print(df["class"].value_counts())


### Alternate — treat `?` as its own readable label, or impute the mode

The brief asks for `"u"`. Two other legal choices: a new token `"missing"` (no collision with cup=`u`), or the mode (`b` = bulbous). Trees can learn from an explicit missing level; imputing the mode invents a root that was never observed.


In [ ]:
# YOUR CODE HERE — do not overwrite df; use a copy
df_mode = None
# df_mode["stalk-root"] = ...
print("mode of raw stalk-root (including ?):", None)


## 2. Exploratory data analysis

Plot three countplots, then a temporary factorize encoding, then the **12 × 12** heatmap of the top features by |corr| with `class`.

1. Class balance.
2. Odor vs class (the key feature).
3. Cap-color vs class (weak on its own).
4. `pd.factorize` every column into integers.
5. Absolute correlation with `class`, drop the target, take `.head(12)`, heatmap **those 12 only**.

Reference images (if you want a visual target): `mushedib_class_balance.png`, `mushedib_odor.png`, `mushedib_capcolor.png`, `mushedib_heatmap.png`.


In [ ]:
# YOUR CODE HERE
# 1. class countplot
# 2. odor vs class
# 3. cap-color vs class
# 4. df_encoded = factorize copy
# 5. correlations / top12 / heatmap of exactly 12 columns
top12 = None
print(top12)


### What you should see

- Balance is close: **51.8% edible / 48.2% poisonous**. Accuracy is a usable headline, but the *costly* error is still a false edible.
- Odor nearly partitions the table. Almond (`a`) and anise (`l`) are edible; foul / fishy / spicy / pungent / creosote / musty are poisonous. `n` (none) is mixed — that is where the other columns earn their keep.
- Cap color overlaps heavily. Brown / yellow / white appear in both classes.
- Factorize |corr| ranking is indicative, not gospel (the codes are unordered). Typical leaders: **odor, spore-print-color, ring-type, stalk-surface-*, gill-size, bruises**.


## 3. Preprocessing

Drop `veil-type`. LabelEncode **every remaining column including `class`**. Split `X` / `y`. `train_test_split(..., test_size=0.2, random_state=42)`. Print the four shapes.

Expected shapes: `X_train (6499, 21)`, `X_test (1625, 21)`.


In [ ]:
# YOUR CODE HERE
# drop veil-type
# encode all columns with LabelEncoder
X = None
y = None
X_train = X_test = y_train = y_test = None
print("X_train shape:", None)


### Alternate — one-hot instead of LabelEncoder

Keep this for logistic regression / kNN later. Do not replace `X_train` used by the forest.


In [ ]:
# YOUR CODE HERE
X_oh = None
print("one-hot width:", None)


## 4. Random Forest

Initialize `RandomForestClassifier(random_state=42)`, fit on the training fold, predict `X_test` into `y_pred`. Name the model `clf` so the evaluation cell matches the brief.


In [ ]:
# YOUR CODE HERE
clf = None
y_pred = None


## 5. Model evaluation

Print accuracy, the numeric confusion matrix, and the classification report (readable names if you want). Heatmap the matrix. Horizontal bar of the **top 5** Gini importances.

On this seed the default forest is typically **perfect** on the 1,625-row test fold: accuracy 1.0, off-diagonal zeros. That is a property of *this table*, not a license to eat wild mushrooms.


In [ ]:
# YOUR CODE HERE
accuracy = None
cm = None
print("Accuracy score:", accuracy)
print("Confusion matrix:\n", cm)
# heatmap
# top-5 importance bar
importances = None


## 6. Alternate code that reaches the same decision

Fit a single `DecisionTreeClassifier`, a one-hot `LogisticRegression`, an odor-majority rule, and a mutual-information ranking. On this table DT and one-hot LR also hit 1.0; the odor rule sits near **0.985** because odor=`n` is mixed.


In [ ]:
# YOUR CODE HERE
dt = None
lr = None
odor_acc = None
mi = None
print("DT acc", None)
print("LR acc", None)
print("odor-rule acc", odor_acc)
print(mi)


## 7. More practice

Work three small drills.

**A.** Restrict the test fold to `habitat == d` (woods) — rebuild a tiny pipeline from the raw frame, or filter the already-encoded rows if you kept a habitat column. Does accuracy stay perfect?

**B.** Cost matrix. Treat a false edible as 10× worse than a false poisonous. Using `predict_proba`, sweep a threshold that flags poisonous more aggressively. How many extra false poisonous do you buy to drive false edibles to zero?

**C.** A 2-feature card: `odor` + `spore-print-color` only. Compare test accuracy and the costly-cell count to the full 21-feature forest.


In [ ]:
# YOUR CODE HERE
# A habitat slice
# B threshold sweep on predict_proba[:, 1]
# C two-feature forest
print("practice A/B/C")


## 8. Simulation — twist a few knobs

Default RF is already perfect, so accuracy-vs-`n_estimators` is a flat line. The knobs that actually move:

| Knob | What we change | What usually happens on this table |
|------|----------------|------------------------------------|
| `max_depth` | 1 → None | depth 1 ≈ 0.87; depth 8 = 1.0 |
| drop features | remove odor / gill-size / spore-print | still ~1.0 until you keep *only* odor (~0.985) |
| label flip | flip 0–35% of *train* labels | test acc holds until ~10%, then falls |
| training n | 50, 100, …, 6,499 | 50 rows ≈ 0.94; 800 rows already ~1.0 |

Edit the four lists, re-run, read the 2×2 panel. A saved reference lives at `mushedib_simulation.png`.


In [ ]:
# YOUR CODE HERE — change these four lists
DEPTHS = [1, 2, 3, 4, 5, 6, 8, None]
FLIP_RATES = [0.0, 0.02, 0.05, 0.10, 0.20, 0.35]
TRAIN_NS = [50, 100, 200, 400, 800, 1600, 3200, len(X_train)]
# then loop, plot, print


## 9. Audience notes (rewrite the same result four ways)

Use the attached audience PDFs (*What to Consider When Considering the Audience*, *Audience and Situation Analysis*). Same numbers, four pitches.

| Audience | Data literacy | Subject knowledge | Time span | What to show |
|----------|---------------|-------------------|-----------|--------------|
| Expert (mycologist / ML) | high | high | long | odor split, MI vs Gini, why 1.0 is not external validity, `u` vs cup collision |
| Technician (app / field key) | medium | high practical | short | the 2-feature card, the costly cell, “do not ship as eat/don’t-eat” |
| Executive (park / food safety) | low–medium | low–medium | very short | 51.8 / 48.2 balance, 0 false edibles on hold-out, **cannot** replace a field guide |
| Nonspecialist | low | low | short | “smell is the giveaway in *this* book of drawings, not in your backyard” |

Full prose: `MushEdib_Project_Memo.docx`.



## 10. Good fit vs limitations

**Good fit**
- All-categorical field-guide codebook with a nearly balanced binary target.
- Tree ensembles and even a shallow DT — splits on odor / spore print / gill size.
- Teaching clean vs mode-impute, LabelEncoder vs one-hot, costly-error thinking.

**Limitations / anti-applications**
- 1987 North-American field-guide codes. New species, look-alikes, regional variants, and photo-only inputs are out of scope.
- `?` → `u` collides with the official cup code.
- Perfect hold-out accuracy is a dataset artifact (features were collected *to* separate edibility).
- Never a foraging decision, never a restaurant receiving spec, never a poisoning-triage tool.

Top applications of the *pattern* (categorical RF + costly FN): defect flags, fraud typology codes, wildlife ID from field marks — always with a human in the loop.

